# Targeting vs non-targeting single-read analysis (one-pot DiMeLo)

`barcode17` and `barcode18` are two antibody conditions from one one-pot DiMeLo run. We:
1. **Extract** per-read 6mA/5mCpG windows over CTCF peaks, non-peak controls, and random sites.
2. **Visualize** single reads (with sorting + auto opacity) and aggregate pileups.
3. Decide which dataset is **targeting** from on-peak mA enrichment.
4. Train a **targeting vs non-targeting** read classifier and inspect it (confusion classes).
5. Add **5mCpG** and compare.
6. Quantify **binding strength = fraction of reads bound** with a peak-methylation-density bound classifier,
   and validate it against random genomic sites (with a centromere-aware karyotype).


In [ ]:
%matplotlib inline
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import pysam
from dimelo import parse_bam, plot_reads, cluster

# --- global plot style (light grid, no top/right spines, readable font) ---
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.25,
                     'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})

# --- file locations ---------------------------------------------------------
data = Path('dimelo/test/data')            # small demo BEDs live in the repo
out = Path('dimelo/test/output')           # large staged files (gitignored)
out.mkdir(exist_ok=True)
ref = out / 'chm13v2.0.fasta'              # barcode17/18 are aligned to CHM13v2.0
chrom_sizes = out / 'chm13v2.0.fasta.fai'  # 2-col chrom/length (for the karyotype)
censat_bed = Path('/oak/stanford/groups/altemose/rosajlee/references/chm13v2.0_censat_v2.1.bed')
peak = data / 'ctcf_demo_peak.bed'         # CTCF ChIP peaks  = real binding sites
notpeak = data / 'ctcf_demo_not_peak.bed'  # non-peak controls
randbed = out / 'random_sites.bed'         # genome-wide random 2 kb windows
# barcode17/18 reads were subset from the full one-pot BAMs over peak+notpeak+random and
# had their MM/ML tags standardized with `modkit update-tags`.
datasets = {'barcode17': out / 'barcode17.onepot.updated.bam',
            'barcode18': out / 'barcode18.onepot.updated.bam'}
MOTIF_COLORS = {'A,0': 'tab:blue', 'CG,0': 'tab:orange'}   # 6mA = blue, 5mCpG = orange
WIN = 1000   # half-window: plots/features span +/- WIN bp around each site center

# --- helpers used throughout -------------------------------------------------
def smooth_bp(y, win=50):
    """50 bp centered moving-average to de-noise a pileup profile (1 position = 1 bp)."""
    y = np.nan_to_num(np.asarray(y, dtype=float))
    return np.convolve(y, np.ones(win) / win, mode='same')

def peak_intervals(bed):
    """Read a BED into {chrom: [(start, end), ...]} for fast overlap tests."""
    iv = {}
    for ln in open(bed):
        f = ln.split()
        if len(f) >= 3:
            iv.setdefault(f[0], []).append((int(f[1]), int(f[2])))
    return iv

def auto_alpha(n_reads, mod_frac):
    """Pick a scatter opacity that fairly represents the data.
    More reads (compressed vertically) and higher modification density => more points stack on
    each other, so we lower alpha; sparse data gets higher alpha so single marks stay visible.
    `overlap` ~ (reads packed into ~300 px) x (fraction of positions modified)."""
    overlap = (max(int(n_reads), 1) / 300.0) * max(float(mod_frac), 1e-3)
    return float(np.clip(0.9 / (1.0 + 20.0 * overlap), 0.05, 0.9))

def centromere_bands(bed=censat_bed):
    """Per-chromosome centromere/constriction extent = span of the CHM13v2.0 cenSat annotation.
    Returns {chrom: (start, end)} which we draw as a pinch on the karyotype ideogram."""
    lo, hi = {}, {}
    if not Path(bed).exists():
        return {}
    for ln in open(bed):
        f = ln.split('\t')
        if len(f) < 3:
            continue
        c, s, e = f[0], int(f[1]), int(f[2])
        lo[c] = min(lo.get(c, s), s); hi[c] = max(hi.get(c, e), e)
    return {c: (lo[c], hi[c]) for c in lo}
def plot_confusion_sorted(Xmat, predictions, positive, split='test'):
    """Single reads grouped by confusion class (TP/TN/FP/FN), with the LARGEST class on top and a
    shared y-scale on the per-class pileups. plot_cluster_profiles orders classes alphabetically, so
    we prefix each label with a size rank ('#1'=biggest) to force largest-first ordering."""
    from collections import Counter
    pr = predictions[predictions['split'] == split]
    rows = pr['row_index'].to_numpy()
    tp = pr['true_label'].to_numpy() == positive; pp = pr['pred_label'].to_numpy() == positive
    lab = np.where(tp & pp, 'TP', np.where(~tp & ~pp, 'TN', np.where(pp, 'FP', 'FN')))
    cnt = Counter(lab); rank = {c: i for i, c in enumerate(sorted(cnt, key=lambda k: -cnt[k]))}
    disp = np.array([f'#{rank[c] + 1} {c}' for c in lab])   # plot adds the (n=...) count itself
    cluster.plot_cluster_profiles(Xmat[rows], disp, view_window_size=WIN, smoothing='gaussian',
        smooth_win=51, point_size=2.0, point_alpha=0.06, cmap_name='coolwarm')
    fig = plt.gcf(); prof_axes = fig.axes[1:]
    peaks = [np.nanmax(l.get_ydata()) for a in prof_axes for l in a.get_lines() if l.get_ydata().size]
    ymax = (max(peaks) if peaks else 0.05) * 1.12
    for a in prof_axes:
        a.set_ylim(0, ymax)
    plt.show()
CEN = centromere_bands()
print('reference:', ref, '| centromere annotations for', len(CEN), 'chromosomes')


## 1. Extract per-read modification windows
`parse_bam.extract` runs modkit and writes an HDF5 of per-read, per-position modification calls
over the requested regions. One extract per dataset covers peaks + non-peak + random windows.


In [ ]:
parsed = {}
for name, bam in datasets.items():
    # override_checks: these are subset reads whose first 100 reads trip a strict alignment QC.
    efile, _ = parse_bam.extract(input_file=bam, output_name=f'{name}_extract', ref_genome=ref,
        output_directory=out, regions=[peak, notpeak, randbed], motifs=['A,0', 'CG,0'],
        thresh=190, window_size=WIN, cores=None, quiet=True, override_checks=True)
    parsed[name] = efile           # path to that dataset's read-level HDF5
    print(f'{name}: extract -> {efile}')


## 2. Single reads over CTCF peaks (motif overlay, sorting, auto opacity)
`plot_reads` draws one dot per modified position, one row per read, both motifs on the same axes
(6mA blue, 5mCpG orange). We first pick an opacity automatically from the data, then show how the
read **ordering** (`sort_by`) changes what structure is visible.


In [ ]:
# helper: load a per-read binary matrix (reads x position) for one motif over some regions,
# orientation-corrected so all regions read 5'->3' (negative-strand regions are flipped).
def motif_matrix(efile, motif, regions):
    rw = cluster.extract_read_windows(hdf5_file=str(efile), motifs=[motif], regions=str(regions),
        config=cluster.ReadWindowExtractionConfig(window_size=WIN, orientation_aware=True),
        span_full_window=False)
    M = np.asarray(rw.data_matrix, dtype=float)
    neg = np.array([str(m.get('region_strand', '.')) == '-' for m in rw.metadata])
    if neg.any():
        M[neg] = M[neg, ::-1]      # flip '-' strand regions into a common 5'->3' frame
    return M

# data-driven opacity: use the number of reads and the mean modification density over peaks
first_dataset = list(datasets)[0]   # used only for the sorting demo below
ALPHA = {}
for name in datasets:
    A = motif_matrix(parsed[name], 'A,0', peak); C = motif_matrix(parsed[name], 'CG,0', peak)
    mod_frac = float(np.nanmean(np.concatenate([A.ravel(), C.ravel()]))) if A.size else 0.02
    ALPHA[name] = auto_alpha(A.shape[0], mod_frac)
    print(f'{name}: {A.shape[0]} reads, mean mod {mod_frac:.3f} -> auto alpha {ALPHA[name]:.2f}')


In [ ]:
# overlay both motifs on the same reads, using each dataset's own auto opacity
for name in datasets:
    fig, ax = plt.subplots(figsize=(9, 3.2))
    plt.sca(ax)                                   # plot_reads draws on the current axes
    plot_reads.plot_reads(mod_file_name=parsed[name], regions=peak, motifs=['A,0', 'CG,0'],
        window_size=WIN, thresh=0.5, regions_5to3prime=True,
        sort_by=['chromosome', 'region_start'], s=1.2, alpha=ALPHA[name], palette=MOTIF_COLORS)
    ax.set_title(f'{name}: single reads over CTCF peaks (6mA=blue, 5mCpG=orange)  [alpha={ALPHA[name]:.2f}]')
    ax.grid(False)
    # legend sits in a reserved right margin (subplots_adjust) so it never covers the reads
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title='Mod', markerscale=2, fontsize=8)
    fig.subplots_adjust(left=0.08, right=0.82, bottom=0.16, top=0.88)
    plt.show()


### Same two datasets, one common opacity (fair comparison)
Per-dataset auto-alpha makes each plot look its best, but the two are then not directly comparable.
For a fair side-by-side, use a single common alpha (the lower of the two so neither saturates) — the
sparser dataset then correctly appears lighter, reflecting that it has fewer/less-modified reads.


In [ ]:
common_alpha = min(ALPHA.values())   # use the denser dataset's alpha for both plots
print('common alpha =', round(common_alpha, 2))
for name in datasets:
    fig, ax = plt.subplots(figsize=(9, 3.2))
    plt.sca(ax)
    plot_reads.plot_reads(mod_file_name=parsed[name], regions=peak, motifs=['A,0', 'CG,0'],
        window_size=WIN, thresh=0.5, regions_5to3prime=True,
        sort_by=['chromosome', 'region_start'], s=1.2, alpha=common_alpha, palette=MOTIF_COLORS)
    ax.set_title(f'{name}: single reads (COMMON alpha={common_alpha:.2f})')
    ax.grid(False)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title='Mod', markerscale=2, fontsize=8)
    fig.subplots_adjust(left=0.08, right=0.82, bottom=0.16, top=0.88)
    plt.show()


### How read ordering changes the view
On a **single peak** (few reads, so ordering is obvious), two `sort_by` choices: by **per-read 6mA
density** (stacks the most-methylated reads together, exposing the bound sub-population) and by
**strand**. Sorting only reorders the rows — the data are identical.


In [ ]:
name = first_dataset
# Sorting is only visible with FEW rows, so use a SINGLE well-covered CTCF peak here.
# 'A,0_mod_fraction' = per-read 6mA fraction (computed by the loader); 'strand' = read strand.
_pk = open(peak).readline().split()
single_region = f'{_pk[0]}:{int(_pk[1])}-{int(_pk[2])}'
print('sorting demo region:', single_region)
sort_opts = [('A,0_mod_fraction', 'by 6mA density (per read)'), ('strand', 'by read strand')]
for sb, label in sort_opts:
    fig, ax = plt.subplots(figsize=(9, 3.0))
    plt.sca(ax)
    plot_reads.plot_reads(mod_file_name=parsed[name], regions=single_region, motifs=['A,0', 'CG,0'],
        window_size=WIN, thresh=0.5, regions_5to3prime=True,
        sort_by=sb, s=6, alpha=0.7, palette=MOTIF_COLORS)   # few reads -> big dots, high opacity
    ax.set_title(f'{name} @ {single_region}: reads sorted {label}')
    ax.grid(False)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title='Mod', markerscale=2, fontsize=8)
    fig.subplots_adjust(left=0.08, right=0.82, bottom=0.16, top=0.88)
    plt.show()


## 3. Aggregate pileups (orientation-corrected, 50 bp smoothed)
The pileup is just the per-position **mean** of the single-read matrix (fraction of reads modified
at each offset). Smoothing over 50 bp removes single-base noise so any peak of methylation density at the site is clear.


In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(5.2 * len(datasets), 3.2), sharey=True)
for ax, name in zip(np.atleast_1d(axes), datasets):
    for motif, color in MOTIF_COLORS.items():
        M = motif_matrix(parsed[name], motif, peak)      # reads x position (0/1 calls)
        prof = smooth_bp(np.nan_to_num(M).mean(axis=0))   # mean over reads = pileup, then smooth
        x = np.arange(-len(prof) // 2, len(prof) - len(prof) // 2)
        ax.plot(x, prof, color=color, lw=1.8, label=f'{motif}  (n={M.shape[0]})')
    ax.axvline(0, color='k', lw=0.6, alpha=0.4)          # CTCF site center
    ax.set_title(f'{name}: motif pileup over CTCF peaks'); ax.set_xlabel('position from center (bp)')
    ax.legend(frameon=False, loc='upper right', fontsize=8)
np.atleast_1d(axes)[0].set_ylabel('fraction modified')
plt.tight_layout(); plt.show()


## 4. Which dataset is targeting?
The **targeting** antibody deposits 6mA specifically at bound CTCF sites, so its on-peak / off-peak
6mA enrichment is higher. We assign the labels from the data rather than hard-coding them.


In [ ]:
def mA_frac(efile, regions):
    M = motif_matrix(efile, 'A,0', regions)
    return float(np.nan_to_num(M).mean()) if M.size else 0.0
enr = {}
for name in datasets:
    on, off = mA_frac(parsed[name], peak), mA_frac(parsed[name], notpeak)
    enr[name] = (on, off, on / max(off, 1e-9))
    print(f'{name}: on-peak mA={on:.3f} off-peak mA={off:.3f} enrichment={enr[name][2]:.2f}x')
targeting = max(enr, key=lambda n: enr[n][2])          # higher on/off enrichment = targeting
non_targeting = [n for n in datasets if n != targeting][0]
print(f'\n=> targeting = {targeting} | non-targeting = {non_targeting}')


## 5. Classify targeting vs non-targeting reads
We build per-read features (PCA of the window + autocorrelation + **peak methylation-density features**, via
`use_peak_features=True`), label each read by its dataset, and train an XGBoost classifier. The
held-out ROC-AUC says how separable the two conditions are on single molecules.


In [ ]:
# per-read feature matrix over CTCF peaks, one row per read; keep metadata + raw matrix aligned
def read_windows(name):
    return cluster.extract_read_windows(hdf5_file=str(parsed[name]), motifs=['A,0'], regions=str(peak),
        config=cluster.ReadWindowExtractionConfig(window_size=WIN, orientation_aware=True),
        span_full_window=False)
rw = cluster.merge_read_window_results([read_windows(targeting), read_windows(non_targeting)],
    source_labels=['targeting', 'non_targeting'], align='error')
MVF = 0.05   # require >=5% of window positions to have a call, else drop the read
feat, feat_names = cluster.read_window_feature_matrix(rw, n_pca=6, use_peak_features=True,
    require_nonzero_valid=True, min_valid_fraction=MVF)
# reproduce the same row filter on the metadata / raw matrix so everything stays aligned to `feat`
meta = pd.DataFrame(rw.metadata)
vs = np.asarray(rw.val_matrix).sum(axis=1); mask = vs > 0
if rw.val_matrix.shape[1] > 0:
    mask &= (vs / rw.val_matrix.shape[1]) >= MVF
meta = meta.loc[mask].reset_index(drop=True)
labels = meta['source_label'].to_numpy()
Xmat = np.asarray(rw.data_matrix)[mask]            # raw 0/1 windows, aligned to feat rows
print('feature matrix:', feat.shape, '| class counts:', dict(zip(*np.unique(labels, return_counts=True))))

clf = cluster.classify_read_features_binary(feat, sample_labels=labels, classifier='xgboost', random_state=42)
print('TEST  accuracy=%.3f roc_auc=%.3f' % (clf['metrics']['test']['accuracy'], clf['metrics']['test']['roc_auc']))
cluster.plot_confusion_matrices(clf['predictions']); plt.show()   # train/test confusion counts


## 6. Test reads grouped by confusion class, with commonly-scaled pileups
Left: each held-out read's 6mA calls, grouped into **TP / TN / FP / FN**. Right: each group's mean
pileup, forced onto a shared y-scale so classes are directly comparable — only true positives
(correctly-called targeting reads) carry a central peak of methylation density.


In [ ]:
plot_confusion_sorted(Xmat, clf['predictions'], 'targeting')   # largest confusion class on top


## 7. Identify true targeting signal
The classifier gives each read a probability of being targeting. Reads in the targeting sample
with high P carry the real peak-methylation-density signal; low-P reads are background even within that sample.


In [ ]:
# orient the classifier probability so it means P(targeting), then map it back per read
pred = clf['predictions'].copy()
pos_is_targeting = pred.loc[pred['pred_label'] == 'targeting', 'proba'].ge(0.5).mean() > 0.5
pred['p_targeting'] = pred['proba'] if pos_is_targeting else 1 - pred['proba']
p_row = np.full(feat.shape[0], np.nan)
p_row[pred['row_index'].to_numpy()] = pred['p_targeting'].to_numpy()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
# (left) distribution of P(targeting), split by which dataset each read came from
for lab, color in [('targeting', 'tab:blue'), ('non_targeting', '0.6')]:
    a1.hist(p_row[labels == lab], bins=20, range=(0, 1), alpha=0.6, color=color, label=lab)
a1.axvline(0.5, color='k', ls='--', lw=1); a1.set_xlabel('P(targeting)'); a1.set_ylabel('reads')
a1.legend(frameon=False, loc='upper center'); a1.set_title('Per-read targeting probability')
# (right) mean 6mA of confident vs background reads within the targeting sample
is_t = labels == 'targeting'; hi = is_t & (p_row >= 0.8); lo = is_t & (p_row < 0.5)
x = np.arange(-Xmat.shape[1] // 2, Xmat.shape[1] - Xmat.shape[1] // 2)
if hi.sum(): a2.plot(x, smooth_bp(np.nan_to_num(Xmat[hi]).mean(0)), color='tab:blue', lw=1.8, label=f'P>=0.8 (n={int(hi.sum())})')
if lo.sum(): a2.plot(x, smooth_bp(np.nan_to_num(Xmat[lo]).mean(0)), color='0.6', lw=1.6, label=f'P<0.5 (n={int(lo.sum())})')
a2.axvline(0, color='k', lw=0.6, alpha=0.4); a2.set_xlabel('position from CTCF center (bp)')
a2.set_ylabel('mean 6mA'); a2.legend(frameon=False, loc='upper right'); a2.set_title(f'True targeting signal in {targeting}')
plt.tight_layout(); plt.show()
print(f'{int(hi.sum())} high-confidence targeting reads of {int(is_t.sum())} in {targeting}.')


## 8. Does adding 5mCpG help the classifier?
`build_multimotif_read_windows` concatenates a 6mA window and a 5mCpG window per read, so the
classifier sees both. We compare it head-to-head with the 6mA-only model from section 5.


In [ ]:
def multi_windows(name):
    return cluster.build_multimotif_read_windows(hdf5_file=str(parsed[name]), motifs=['A,0', 'CG,0'],
        regions=str(peak), window_size=WIN, orientation_aware=True, span_full_window=False,
        require_all_motifs=True)
rw_m = cluster.merge_read_window_results([multi_windows(targeting), multi_windows(non_targeting)],
    source_labels=['targeting', 'non_targeting'], align='error')
feat_m, _ = cluster.read_window_feature_matrix(rw_m, n_pca=8, use_peak_features=True,
    require_nonzero_valid=True, min_valid_fraction=MVF)
meta_m = pd.DataFrame(rw_m.metadata)
vsm = np.asarray(rw_m.val_matrix).sum(axis=1); mm = vsm > 0
if rw_m.val_matrix.shape[1] > 0:
    mm &= (vsm / rw_m.val_matrix.shape[1]) >= MVF
labels_m = meta_m.loc[mm, 'source_label'].to_numpy()
clf_m = cluster.classify_read_features_binary(feat_m, sample_labels=labels_m, classifier='xgboost', random_state=42)
# bar chart comparing test performance of 6mA-only vs 6mA+5mCpG
models = ['6mA only', '6mA + 5mCpG']
acc = [clf['metrics']['test']['accuracy'], clf_m['metrics']['test']['accuracy']]
auc = [clf['metrics']['test']['roc_auc'], clf_m['metrics']['test']['roc_auc']]
x = np.arange(2); w = 0.35
fig, ax = plt.subplots(figsize=(4.8, 3.2))
b1 = ax.bar(x - w / 2, acc, w, label='accuracy', color='tab:blue')
b2 = ax.bar(x + w / 2, auc, w, label='ROC-AUC', color='tab:orange')
ax.set_xticks(x); ax.set_xticklabels(models); ax.set_ylim(0, 1.08)
ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1.0))
ax.set_title('Test performance: single- vs multi-motif')
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()


## 9. Binding strength = fraction of reads bound (site-wise single-read classification)
On a single molecule a read is either **bound** (peak of methylation density at the site) or not, so a
site's binding strength is the **fraction of its reads that are bound**. We classify each single read
bound/unbound with the same XGBoost model as section 5, but trained on **location** labels — bound =
read over a CTCF peak, unbound = read over a random window — using peak methylation-density features.
Location only supervises training; at scoring time a read is called from its own signal, and we
validate on held-out sites.


In [ ]:
from sklearn.metrics import roc_auc_score
# reads over CTCF peaks (bound) + random windows (unbound), from both datasets, one feature matrix
rws, src = [], []
for nm in (targeting, non_targeting):
    for reg, lab in [(peak, 'bound'), (randbed, 'unbound')]:
        rws.append(cluster.extract_read_windows(hdf5_file=str(parsed[nm]), motifs=['A,0'], regions=str(reg),
            config=cluster.ReadWindowExtractionConfig(window_size=WIN, orientation_aware=True), span_full_window=False))
        src.append(lab)                        # label a read by WHERE it is (peak vs random)
rw_v = cluster.merge_read_window_results(rws, source_labels=src, align='error')
feat_v, _ = cluster.read_window_feature_matrix(rw_v, n_pca=6, use_peak_features=True,
    require_nonzero_valid=True, min_valid_fraction=MVF)
meta_v = pd.DataFrame(rw_v.metadata)
vv = np.asarray(rw_v.val_matrix).sum(axis=1); mv = vv > 0
if rw_v.val_matrix.shape[1] > 0:
    mv &= (vv / rw_v.val_matrix.shape[1]) >= MVF
meta_v = meta_v.loc[mv].reset_index(drop=True)
Xv = np.asarray(rw_v.data_matrix)[mv]
labels_bound = meta_v['source_label'].to_numpy()                       # 'bound' (peak) / 'unbound' (random)
meta_v['source_site'] = np.where(labels_bound == 'bound', 'real', 'random')
meta_v['region_id'] = (meta_v['chromosome'].astype(str) + ':' + meta_v['region_start'].astype(int).astype(str)
                       + '-' + meta_v['region_end'].astype(int).astype(str))
clf_site = cluster.classify_read_features_binary(feat_v, sample_labels=labels_bound, classifier='xgboost', random_state=42)
print('site-wise bound classifier TEST accuracy=%.3f roc_auc=%.3f'
      % (clf_site['metrics']['test']['accuracy'], clf_site['metrics']['test']['roc_auc']))
cluster.plot_confusion_matrices(clf_site['predictions']); plt.show()


### Held-out reads grouped by confusion class (largest class on top)
Each held-out read's 6mA calls, grouped into TP / TN / FP / FN with the per-class pileups on a shared
scale. TP = reads correctly called bound; their pileup shows the central peak of methylation density.


In [ ]:
plot_confusion_sorted(Xv, clf_site['predictions'], 'bound')


### Per-site fraction of reads bound (validated on held-out sites)
Turn per-read P(bound) into a per-site fraction of reads bound. AUROC is on held-out reads; the map
below uses all reads.


In [ ]:
def per_site_fraction(clf, positive, only_test=True):
    """Per-site fraction of reads called bound (P>=0.5). only_test restricts to held-out reads."""
    pr = clf['predictions'].copy()
    pos = pr.loc[pr['pred_label'] == positive, 'proba'].ge(0.5).mean() > 0.5      # orient proba to P(bound)
    pr['p'] = pr['proba'] if pos else 1 - pr['proba']
    p = np.full(feat_v.shape[0], np.nan); sp = np.array([''] * feat_v.shape[0], dtype=object)
    p[pr['row_index'].to_numpy()] = pr['p'].to_numpy(); sp[pr['row_index'].to_numpy()] = pr['split'].to_numpy()
    d = meta_v.assign(p=p, split=sp).dropna(subset=['p'])
    if only_test:
        d = d[d['split'] == 'test']
    s = (d.groupby(['region_id', 'chromosome', 'region_start', 'region_end', 'source_site'])
         .agg(n_reads=('p', 'size'), n_bound=('p', lambda s: int((s >= 0.5).sum()))).reset_index())
    s = s[s['n_reads'] >= 3].copy(); s['binding_strength'] = s['n_bound'] / s['n_reads']
    return s
sites_ho = per_site_fraction(clf_site, 'bound', only_test=True)     # held-out sites -> AUROC
sites = per_site_fraction(clf_site, 'bound', only_test=False)       # all sites -> genome map
auc = roc_auc_score((sites_ho['source_site'] == 'real').astype(int), sites_ho['binding_strength']) \
      if sites_ho['source_site'].nunique() > 1 else float('nan')
print('mean fraction of reads bound per site (all sites):')
print(sites.groupby('source_site')['binding_strength'].agg(['count', 'mean', 'median']))
print(f'\nAUROC (fraction bound separates real CTCF vs random, held-out sites) = {auc:.3f}')


### Sites sorted by source, then by fraction of reads bound


In [ ]:
so = sites.sort_values(['source_site', 'binding_strength'], ascending=[False, True]).reset_index(drop=True)
col = so['source_site'].map({'real': 'tab:green', 'random': '0.6'})
fig, ax = plt.subplots(figsize=(6.5, 7.0))
ax.barh(np.arange(len(so)), so['binding_strength'], color=col, height=1.0)
ax.axhline(int((so['source_site'] == 'real').sum()) - 0.5, color='k', lw=1.2)
ax.set_xlim(0, 1); ax.set_yticks([]); ax.set_xlabel('binding strength = fraction of reads bound')
ax.set_ylabel('site  (grouped by source, sorted by strength)'); ax.grid(axis='x', alpha=0.3)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='tab:green', label='real (CTCF peak)'), Patch(color='0.6', label='random')],
          frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1.0))
ax.set_title(f'Per-site fraction of reads bound  (held-out AUROC = {auc:.2f})')
plt.tight_layout(); plt.show()


### Binding strength across the genome
Fraction of reads bound mapped onto the karyotype, with a red sphere at each centromere. Sites are
circles (real CTCF peak) / triangles (random).


In [ ]:
import re
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
sz = {l.split()[0]: int(l.split()[1]) for l in open(chrom_sizes)}
def _ck(c):
    mm = re.match(r'chr(\d+)$', str(c))
    return (0, int(mm.group(1))) if mm else (1, {'chrX': 23, 'chrY': 24, 'chrM': 26}.get(str(c), 25))
chroms = sorted(sites['chromosome'].unique(), key=_ck)
xmap = {c: i for i, c in enumerate(chroms)}
cmap = plt.get_cmap('viridis'); norm = Normalize(0, 1)
fig, ax = plt.subplots(figsize=(min(15, 2 + len(chroms) * 0.7), 7))
for c in chroms:
    x = xmap[c]
    ax.plot([x, x], [0, sz[c]], color='0.88', lw=8, solid_capstyle='round', zorder=1)   # chromosome body
    if c in CEN:                                                                          # centromere = red sphere
        cs, ce = CEN[c]
        ax.scatter([x], [(cs + ce) / 2], marker='o', s=300, facecolor='red', edgecolor='darkred', linewidth=0.8, zorder=4)
for src, marker in [('real', 'o'), ('random', '^')]:
    sub = sites[sites['source_site'] == src]
    if not len(sub):
        continue
    ax.scatter([xmap[c] for c in sub['chromosome']], (sub['region_start'] + sub['region_end']) / 2,
               c=sub['binding_strength'], cmap=cmap, norm=norm, marker=marker, s=48,
               edgecolor='k', linewidth=0.3, zorder=3)
ax.set_xticks(range(len(chroms))); ax.set_xticklabels(chroms, rotation=90)
ax.invert_yaxis(); ax.set_yticks([])
for s in ('top', 'right', 'bottom', 'left'):
    ax.spines[s].set_visible(False)
ax.grid(False)
cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, fraction=0.025, pad=0.02)
cb.set_label('fraction of reads bound')
handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor='0.5', markeredgecolor='k', label='real (CTCF peak)'),
           Line2D([0], [0], marker='^', color='w', markerfacecolor='0.5', markeredgecolor='k', label='random'),
           Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markeredgecolor='darkred', markersize=11, label='centromere')]
ax.legend(handles=handles, frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.10), ncol=3, title='site source')
ax.set_title('CTCF sites on the karyotype colored by fraction of reads bound')
plt.tight_layout(); plt.show()


### Interpreting the result
Binding strength is a **single-molecule fraction of reads bound**: the site-wise classifier calls each
read bound/unbound from its peak methylation density, and the site value is the fraction called bound.
Trained on peak-vs-random (weak, positional labels) it learns the local density signature and
separates real CTCF sites from random ones on held-out data. CTCF is only an example here — a factor
with weaker local enrichment would simply show a lower fraction of reads bound.
